In [26]:
import os
import re 
import cv2
import numpy as np
import pandas as pd

from PIL import Image
from typhoon_ocr import ocr_document

from sklearn.model_selection import train_test_split, cross_validate
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier


In [27]:
API_KEY = "sk-HytrxVrL2v19alxMtEs8BzXfGpzTWX5Uc7x6q4mYR9osAcYl"  

def extract_text(image_path):
    return ocr_document(image_path, api_key=API_KEY)


In [28]:
def normalize(text):
    return re.sub(r"\s+", " ", text.lower())

In [29]:
MFU_KEYWORDS = [
    "mae fah luang university",
    "333 moo 1",
    "thasud",
    "muang",
    "chiang rai",
    "57100",
    "0994000165178"
]

In [30]:
def extract_header_block(text):
    text = normalize(text)   # normalize first
    lines = text.split("\n")
    header = []
    
    for line in lines:
        if any(k in line for k in [
            "description", "qty", "price", "quantity",
            "unit price", "subtotal", "item", "product",
            "no.", "amount", "total"
        ]):
            break
        
        header.append(line.strip())
    
    return header


In [31]:
def extract_footer_block(text):
    lines = text.split("\n")
    footer = []
    
    for line in reversed(lines):
        low = line.lower()
        footer.insert(0, line.strip())
        
        if any(k in low for k in ["grand total", "total amount"]):
            break
    
    return footer


In [32]:
def is_mfu_address_line(line):
    low = line.lower()
    return any(k in low for k in [
        "mae fah luang university",
        "333 moo 1",
        "thasud",
        "muang",
        "chiang rai",
        "57100",
        "0994000165178"
    ])



In [33]:
MONTHS = (
    "jan|january|feb|february|mar|march|apr|april|may|"
    "jun|june|jul|july|aug|august|sep|september|"
    "oct|october|nov|november|dec|december"
)

def detect_date(text):
    t = normalize(text)
    patterns = [
        r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b",
        rf"\b\d{{1,2}}\s+({MONTHS})\s+\d{{4}}\b",
        rf"\b({MONTHS})\s+\d{{1,2}},?\s+\d{{4}}\b"
        rf"\b\d{{1,2}}\s+({MONTHS})\s+\d{{4}}\b", 
        rf"\b({MONTHS})\s+\d{{1,2}},?\s+\d{{4}}\b"
    ]
    return any(re.search(p, t) for p in patterns)


In [34]:
def detect_store_name(text):
    STORE_KEYWORDS = [
        "co.", "ltd", "limited", "company", "corp",
        "shop", "store", "hotel",
        "studies", "handicraft", "flower",
        "supply", "book", "coffee"
]


    t = text.lower()

    # If ANY store keyword exists → store name exists
    if any(re.search(rf"\b{k}\b", line.lower()) for k in STORE_KEYWORDS for line in text.split("\n")):
        return True


    # Otherwise → missing store name
    return False




In [35]:
import re

def detect_store_address(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    low_lines = [l.lower() for l in lines]

    customer_block = set()


    for i, low in enumerate(low_lines):
        if any(x in low for x in [
            "bill to", "billed to",
            "received from"
        ]):
            for j in range(i, min(i + 8, len(lines))):
                customer_block.add(j)

   
    address_keywords = [
        "road", "rd", "street", "st", "soi", "moo",
        "district", "subdistrict", "muang",
        "province", "city",
        "bangkok", "chiang rai", "thailand"
    ]

    def is_real_address(line):
        has_number = bool(re.search(r"\d+", line))
        has_keyword = any(k in line for k in address_keywords)
        return has_number and has_keyword


    for i in range(len(lines)):
        if i not in customer_block:
            if is_real_address(low_lines[i]):
                return True

    return False



In [36]:
def detect_item_list(text):
    lines = text.split("\n")

    count = 0
    for line in lines:
        if re.search(r"\d+[.,]\d{2}", line):
            count += 1

    return count >= 1

In [37]:
def detect_amount_in_words(text):
    t = text.lower()
    return (
        "baht only" in t
        or "amount in words" in t
        or "satang" in t
    )


In [38]:
def detect_signature(text):
    lines = [l.strip() for l in text.split("\n") if l.strip()]
    low_lines = [l.lower() for l in lines]

    signature_keywords = [
        "signature", "signed by", "approved by", "received by"
    ]

    roles = [
        "manager", "cashier", "owner", "authorized", "director"
    ]

    # 1️⃣ Explicit signature keywords
    if any(any(k in low for k in signature_keywords) for low in low_lines):
        return True

    bottom_start = int(len(low_lines) * 0.6)
    for low in low_lines[bottom_start:]:
        if low in roles:
            return True

    return False



In [39]:
from PIL import Image
import os

def crop_bottom_for_signature(image_path, ratio=0.4):
    img = Image.open(image_path)
    w, h = img.size

    # bottom 40% (good default for receipts)
    crop_box = (0, int(h * (1 - ratio)), w, h)
    cropped = img.crop(crop_box)

    tmp_path = "tmp_signature.jpg"
    cropped.save(tmp_path)
    return tmp_path


In [40]:
import cv2
import numpy as np

def detect_handwritten_signature(image_path):
    img = cv2.imread(image_path, cv2.IMREAD_GRAYSCALE)
    if img is None:
        return False

    blur = cv2.GaussianBlur(img, (5, 5), 0)

    thresh = cv2.adaptiveThreshold(
        blur, 255,
        cv2.ADAPTIVE_THRESH_MEAN_C,
        cv2.THRESH_BINARY_INV,
        15, 5
    )

    ink_pixels = np.sum(thresh > 0)
    total_pixels = thresh.size

    ink_ratio = ink_pixels / total_pixels

    return ink_ratio > 0.01



In [41]:
def analyze_receipt(text):
    """Main analysis function"""
    missing = []
    
    if not detect_date(text):
        missing.append("Date")
    
    if not detect_amount_in_words(text):
        missing.append("Amount in Words")
    
    if not detect_store_name(text):
        missing.append("Store Name")
    
    if not detect_store_address(text):
        missing.append("Store Address")
    
    if not detect_signature(text):
        missing.append("Signature")
    
    return missing

In [42]:
def build_dataset(folder, label):
    rows = []

    for img in os.listdir(folder):
        path = os.path.join(folder, img)

        # --- Full OCR ---
        text = extract_text(path)

        # --- Crop bottom for signature ---
        sig_img = crop_bottom_for_signature(path)
        sig_text = extract_text(sig_img)

        # --- Signature detection (TEXT OR IMAGE) ---
        text_sig = detect_signature(sig_text)
        img_sig = detect_handwritten_signature(sig_img)
        has_signature = text_sig or img_sig

        rows.append({
            "has_date": detect_date(text),
            "has_store_name": detect_store_name(text),
            "has_store_address": detect_store_address(text),
            "has_amount_words": detect_amount_in_words(text),
            "has_signature": has_signature,
            "label": label
        })

    return rows



In [43]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.DataFrame(
    build_dataset("data/complete", 1) +
    build_dataset("data/incomplete", 0)
)

X = df.drop("label", axis=1)
y = df["label"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)


In [44]:
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np

models = {
    "Logistic Regression": LogisticRegression(max_iter=1000),
    "Decision Tree": DecisionTreeClassifier(max_depth=5),
    "SVM": SVC(kernel="linear"),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42)
}

cv_results = []

for name, model in models.items():
    f1 = cross_val_score(model, X, y, cv=5, scoring="f1").mean()
    precision = cross_val_score(model, X, y, cv=5, scoring="precision").mean()
    recall = cross_val_score(model, X, y, cv=5, scoring="recall").mean()
    accuracy = cross_val_score(model, X, y, cv=5, scoring="accuracy").mean()

    cv_results.append({
        "Model": name,
        "Accuracy": round(accuracy, 6),
        "Precision": round(precision, 6),
        "Recall": round(recall, 6),
        "F1": round(f1, 6)
    })

cv_results_df = pd.DataFrame(cv_results).set_index("Model")
cv_results_df


,Accuracy,Precision,Recall,F1
Model,,,,
Logistic Regression,0.85,0.900000,0.9,0.866667
Decision Tree,0.95,1.000000,0.9,0.933333
SVM,0.80,0.833333,0.9,0.826667
KNN,0.55,0.533333,0.9,0.660000
Random Forest,0.95,1.000000,0.9,0.933333


In [45]:
best_model_name = cv_results_df["F1"].idxmax()
best_model = models[best_model_name]
best_model.fit(X, y)
best_model


,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",5
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary ` for details.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsamples at the current 

In [46]:
def run_all_receipts(models):
    print("=== COMPLETE RECEIPTS ===")
    for img in sorted(os.listdir("data/complete")):
        path = os.path.join("data/complete", img)

        # --- normal OCR for most fields ---
        text = extract_text(path)

        # --- OCR ONLY bottom part for signature ---
        sig_img = crop_bottom_for_signature(path)
        sig_text = extract_text(sig_img)

        # --- analyze ---
        missing = analyze_receipt(text)

        if detect_signature(sig_text) is False:
            if "Signature" not in missing:
                missing.append("Signature")
        else:
            if "Signature" in missing:
                missing.remove("Signature")

        prediction = "INCOMPLETE" if missing else "COMPLETE"

        print(f"\nReceipt: {img}")
        print("Prediction:", prediction)
        if missing:
            print("Missing fields:", missing)

    print("\n=== INCOMPLETE RECEIPTS ===")
    for img in sorted(os.listdir("data/incomplete")):
        path = os.path.join("data/incomplete", img)

        text = extract_text(path)
        sig_img = crop_bottom_for_signature(path)
        sig_text = extract_text(sig_img)
        signature_exists = detect_signature(sig_text) and detect_handwritten_signature(sig_img)


        missing = analyze_receipt(text)

        if detect_signature(sig_text) is False:
            if "Signature" not in missing:
                missing.append("Signature")
        else:
            if "Signature" in missing:
                missing.remove("Signature")

        prediction = "INCOMPLETE" if missing else "COMPLETE"

        print(f"\nReceipt: {img}")
        print("Prediction:", prediction)
        if missing:
            print("Missing fields:", missing)

In [47]:
run_all_receipts(best_model)


=== COMPLETE RECEIPTS ===

Receipt: C1.jpg
Prediction: COMPLETE

Receipt: C10.jpg
Prediction: COMPLETE

Receipt: C2.jpg
Prediction: COMPLETE

Receipt: C3.jpg
Prediction: COMPLETE

Receipt: C4.jpg
Prediction: COMPLETE

Receipt: C5.jpg
Prediction: INCOMPLETE
Missing fields: ['Amount in Words']

Receipt: C6.jpg
Prediction: COMPLETE

Receipt: C7.jpg
Prediction: COMPLETE

Receipt: C8.jpg
Prediction: COMPLETE

Receipt: C9.jpg
Prediction: COMPLETE

=== INCOMPLETE RECEIPTS ===

Receipt: U1.jpg
Prediction: INCOMPLETE
Missing fields: ['Amount in Words']

Receipt: U10.jpg
Prediction: INCOMPLETE
Missing fields: ['Amount in Words', 'Signature']

Receipt: U2.jpg
Prediction: INCOMPLETE
Missing fields: ['Store Name', 'Store Address']

Receipt: U3.jpg
Prediction: INCOMPLETE
Missing fields: ['Date']

Receipt: U4.jpg
Prediction: INCOMPLETE
Missing fields: ['Date']

Receipt: U5.jpg
Prediction: INCOMPLETE
Missing fields: ['Amount in Words', 'Store Name', 'Store Address']

Receipt: U6.jpg
Prediction: INCOMP

In [48]:
text = extract_text("data/incomplete/U6.jpg")
print(text)

# PAYMENT RECEIPT
NORTHERN SCIENCE SUPPLY LTD.

RECEIPT NO.: 161326
DATE: 21 January 2025
RECEIVED FROM: Mae Fah Luang University
ADDRESS: 333 Moo 1, Thasud Subdistrict, Muang District,
Chiang Rai 57100, Thailand
Tax ID: 0994000165178

<table><tr><td>DESCRIPTION</td><td>QTY.</td><td>UNIT PRICE</td><td>TOTAL</td></tr><tr><td>Nitrile Examination Gloves (Box of 100)</td><td>5</td><td>280.00</td><td>1,400.00</td></tr><tr><td>Glass Beaker 250ml (Heat Resistant)</td><td>10</td><td>150.00</td><td>1,500.00</td></tr><tr><td>Ethanol 95% (Gallon 3.8L)</td><td>2</td><td>450.00</td><td>900.00</td></tr><tr><td>Filter Paper (Pack of 100)</td><td>4</td><td>120.00</td><td>480.00</td></tr></table>

SUB-TOTAL: 4,280.00
DISCOUNT: 280.00
VAT (7%): 280.00
TOTAL: 4,280.00

PAYMENT METHOD: Bank Transfer
TRANSACTION ID: 123-456-7890
NOTES:

Signature

GIRLIE REYES
Cashier
